# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank.ai_internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Content Refresh Prioritization (Declining Content Detection)**

I am choosing the core lane: predicting which content pages are declining in search performance so they can be prioritized for refresh. This lane is chosen because it directly maps to FlyRank's core product — clients need to know which of their pages are losing search visibility before it becomes a serious ranking loss. The label is well-defined (`is_declining_label = trend_direction == 'down'`), the data is rich (44 columns covering GSC, GA4, keyword, and content signals), and the business impact is immediate: a ranked refresh queue that a content team can act on Monday morning.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong call cost?*

**Decision:** Which content pages should a content team refresh first in the coming sprint?

**Who acts:** Content managers / SEO strategists at FlyRank's client companies. They use the model's ranked output to allocate writer time and editorial effort.

**Cost of a wrong call:**
- **False positive (refresh a page that is not actually declining):** Writer time wasted on healthy content; opportunity cost of not refreshing genuinely declining pages.
- **False negative (miss a declining page):** The page continues to lose impressions and clicks. For a high-traffic page, even one week of delay can mean thousands of lost clicks. Google may also downrank the page further, making recovery harder.

**Asymmetry:** False negatives are costlier — a missed declining page keeps bleeding traffic. Precision matters for writer efficiency, but recall is the metric we guard most carefully.

## 3. Three column groups and the signal story

*Pick three groups of columns from the data dictionary. For each: what does it tell you, and why might it predict your outcome?*

**Group 1 — 30-day comparison windows (`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` vs `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`)**

These are the most direct signals of recent momentum. A page whose impressions dropped from prev_30d to last_30d is likely declining. However, `trend_pct` and `trend_direction` are derived from exactly these columns, so I must use the raw 30d windows rather than the derived label-adjacent columns.

**Group 2 — Content properties (`word_count`, `content_age_days`, `days_since_last_update`)**

Old content that has never been updated is a strong predictor of staleness. Google's freshness signals reward recently-updated pages. A page that is 500+ days old and has never been touched is a classic refresh candidate even before its traffic dips.

**Group 3 — Engagement metrics (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`)**

Low CTR at a given position signals a title/meta that is losing the SERP competition. Low engagement_rate and scroll_rate signal that visitors who do arrive are not finding the content satisfying — a quality signal Google can pick up through behavioral data. These metrics together tell whether a page is being seen (impressions) but not clicked (CTR), or clicked but then immediately abandoned (engagement).

## 4. One truthful sentence about what this model *cannot* do

*Honest about limits — what would make this model wrong or misused?*

**This model cannot distinguish between content that is declining because of poor quality versus content that is declining because of an external SERP shift (e.g., Google added a featured snippet or a competitor published a better resource) — in both cases the label fires, but the action the content team should take is completely different.**

Additionally:
- The model is trained on 30,000 pages from 32 anonymized clients. It may not generalize well to client industries or content types not represented in those 32 clients.
- Seasonal dips (holiday slowdowns, industry conference periods) can trigger the declining label for pages that are healthy and will recover on their own.
- The label is computed over a 90-day trailing window — a page that just started declining in the last two weeks may not yet cross the threshold.

In [ ]:
# Quick sanity check — load the data and confirm the lane label distribution
import pandas as pd
import os

# Adjust path if running locally vs Colab
if os.path.exists('data/raw/content_refresh_anonymized.csv'):
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
elif os.path.exists('../data/raw/content_refresh_anonymized.csv'):
    df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')
else:
    raise FileNotFoundError('Run from repo root or Colab with the starter repo cloned.')

print(f'Dataset shape: {df.shape}')
print(f'\nLabel distribution (trend_direction):')
print(df['trend_direction'].value_counts())
print(f'\nDecline rate: {(df["trend_direction"]=="down").mean():.1%}')
print(f'\nNumber of unique clients: {df["client_id"].nunique()}')
print(f'Content types: {df["content_type"].value_counts().to_dict()}')